# Gen Z / Alpha Slang ↔ English Translator — Fine-tuning Notebook

**Group 2** · Llama 3.2 3B Instruct + QLoRA · runs on a **T4 GPU** (free Colab).

This notebook fine-tunes one model that translates **both directions**, chosen by a tag:
- `Translate to English:` → plain English
- `Translate to Gen Z slang:` → slang

**How to run:** `Runtime → Change runtime type → T4 GPU`, then `Runtime → Run all`.
Run the cells top to bottom. Each section is labelled. See `USER_MANUAL.md` for the full walkthrough.

---
### Pipeline
1. Setup (detect Colab/local, install, load project)  
2. Prepare data (build train/eval from `data/`)  
3. Load base model  
4. **Baseline** — generate on the eval set *before* tuning (this is our 'before')  
5. Train (QLoRA)  
6. **Tuned** — generate on the eval set *after* tuning  
7. Export a grading sheet (human raters score base vs tuned)  
8. *(stretch)* Local 8B model as an automatic judge  
9. Try it yourself

## 1 · Setup

In [21]:
# Detect environment: Colab installs packages here; locally we assume the uv env is already set up.
import os, sys
IN_COLAB = 'google.colab' in sys.modules
print('Running in Colab' if IN_COLAB else 'Running locally (expecting the uv environment)')

Running locally (expecting the uv environment)


In [22]:
# Install deps. On Colab this pulls the GPU stack; Unsloth pins compatible torch/transformers/trl.
if IN_COLAB:
    # unsloth picks compatible torch/transformers/trl/peft/bitsandbytes automatically.
    !pip install -q unsloth
    !pip install -q pandas openpyxl datasets
else:
    # Local: the uv environment already has everything (see USER_MANUAL.md).
    print('Local run: using the uv environment (torch+cu128, unsloth, trl, etc.).')

Local run: using the uv environment (torch+cu128, unsloth, trl, etc.).


In [23]:
# Locate the project folder (the one containing the data/ and src/ folders).
from pathlib import Path
if IN_COLAB:
    # Option A: upload+unzip the project zip so /content/genz_slang_prj exists, OR
    # Option B: mount Google Drive and point PROJECT_DIR at the folder there.
    # Edit this to match where you put the project on Colab:
    PROJECT_DIR = Path('/content/genz_slang_prj')
    if not PROJECT_DIR.exists():
        # Fallback: mount Drive (uncomment and set your path)
        # from google.colab import drive; drive.mount('/content/drive')
        # PROJECT_DIR = Path('/content/drive/MyDrive/genz_slang_prj')
        raise FileNotFoundError(
            'Set PROJECT_DIR to your uploaded/mounted project folder (must contain data/ and src/).')
else:
    PROJECT_DIR = Path.cwd()
    if not (PROJECT_DIR / 'src').exists() and (PROJECT_DIR.parent / 'src').exists():
        PROJECT_DIR = PROJECT_DIR.parent

sys.path.insert(0, str(PROJECT_DIR / 'src'))
os.chdir(PROJECT_DIR)
print('PROJECT_DIR =', PROJECT_DIR)
assert (PROJECT_DIR / 'data').exists(), 'data/ folder not found under PROJECT_DIR'

PROJECT_DIR = c:\Users\drama\Downloads\slang prj


In [24]:
# Hugging Face login. We use Unsloth's pre-quantized mirror which is usually open,
# but a free HF token avoids any rate limits / gated-download surprises.
# Get one at https://huggingface.co/settings/tokens (read scope is enough).
if IN_COLAB:
    from huggingface_hub import login
    # Prefer Colab Secrets: add HF_TOKEN under the key icon on the left, then:
    try:
        from google.colab import userdata
        login(userdata.get('HF_TOKEN'))
        print('Logged in to Hugging Face via Colab secret.')
    except Exception as e:
        print('No HF secret found (that is OK for the Unsloth mirror). Detail:', e)

## 2 · Prepare data
Builds `data/processed/train.jsonl` and the frozen `data/processed/eval.jsonl` from `data/raw/`.
Deterministic (seed=42) so the eval set is identical every run. Safe to re-run.

In [25]:
import subprocess
result = subprocess.run([sys.executable, 'src/prepare_data.py'], capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr); raise RuntimeError('Data prep failed')

Loading sources from data/raw/ ...
  ok: genz_dataset.csv                         ->  1005 usable pairs
  ok: genz_dataset_augmented_1500.csv          ->  1515 usable pairs
  ok: genz_synthetic_dataset.csv               -> 14773 usable pairs

Deduped: 17293 -> 15815 unique pairs

Using EXISTING frozen eval.jsonl (70 items). Delete it to re-freeze.
Added 300 abstain (unclear->decline) training examples.

--- SUMMARY ---
train.jsonl : 31852 examples (from 15776 pairs x 2 directions)
eval.jsonl  :    70 items (30 pairs x 2 directions + 10 unanswerable)
  eval by direction: {'to_english': 30, 'to_slang': 30, 'unanswerable': 10}
  eval by strat    : {'real': 20, 'trending': 20, 'viral': 20, 'unanswerable': 10}

Wrote:
  C:\Users\drama\Downloads\slang prj\data\processed\train.jsonl
  C:\Users\drama\Downloads\slang prj\data\processed\eval.jsonl



In [26]:
import json
from config import TRAIN_PATH, EVAL_PATH, TAG_TO_ENGLISH, TAG_TO_SLANG, ABSTAIN_MESSAGE
from abstain import looks_unclear, is_abstention

def read_jsonl(p):
    with open(p, encoding='utf-8') as f:
        return [json.loads(l) for l in f]

train_rows = read_jsonl(TRAIN_PATH)
eval_rows = read_jsonl(EVAL_PATH)
print(f'train examples: {len(train_rows)}   eval items: {len(eval_rows)}')
print('example eval item:', eval_rows[0])

train examples: 31852   eval items: 70
example eval item: {'id': 'e2e_000', 'direction': 'to_english', 'type': 'translate', 'tag': 'Translate to English:', 'input': 'She has such beta af energy today.', 'reference': 'She has such extremely beta energy today.', 'term': 'beta af', 'meaning': 'extremely beta', 'strat': 'real'}


## 3 · Load base model (Llama 3.2 3B Instruct, 4-bit)

In [27]:
from unsloth import FastLanguageModel
import torch

MAX_SEQ_LEN = 1024   # slang sentences are short; keeps T4 memory low
BASE_MODEL = 'unsloth/Llama-3.2-3B-Instruct-bnb-4bit'  # pre-quantized, T4-friendly

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = BASE_MODEL,
    max_seq_length = MAX_SEQ_LEN,
    dtype = None,          # auto (fp16 on T4)
    load_in_4bit = True,
)

from unsloth.chat_templates import get_chat_template
tokenizer = get_chat_template(tokenizer, chat_template='llama-3.1')
print('Base model loaded.')

==((====))==  Unsloth 2026.7.2: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    NVIDIA GeForce RTX 5070. Num GPUs = 1. Max memory: 11.94 GB. Platform: Windows.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 12.0. CUDA Toolkit: 12.8. Triton: 3.7.1
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights: 100%|██████████| 254/254 [00:00<00:00, 304.47it/s]
Unsloth: Will load unsloth/Llama-3.2-3B-Instruct-bnb-4bit as a legacy tokenizer.


Base model loaded.


In [28]:
# Helper: generate a model's answer for one eval item, via the SHARED core
# (translate_core.py) so the eval, app.py and serve.py all decode identically
# (same anti-repetition settings). use_guard=True is for the app/demo; the eval
# loops call with use_guard=False to measure the raw model.
from translate_core import translate as core_translate

def translate(item, use_guard=False):
    FastLanguageModel.for_inference(model)
    direction = 'to_slang' if item['tag']==TAG_TO_SLANG else 'to_english'
    return core_translate(model, tokenizer, item['input'], direction, use_guard=use_guard)

## 4 · Baseline (BEFORE tuning) — this is the 'gap'
Run the untouched base model on all 60 eval items and save its answers. These are the
'before' numbers you compare against on Friday. Look for confidently-wrong slang.

In [29]:
from tqdm.auto import tqdm
for it in tqdm(eval_rows, desc='baseline'):
    it['base_output'] = translate(it)

# Peek at a few, especially where the base model is likely to struggle:
for it in eval_rows[:6]:
    print(f"[{it['direction']}] IN: {it['input']}")
    print(f"   REF : {it['reference']}")
    print(f"   BASE: {it['base_output']}\n")

baseline:   0%|          | 0/70 [00:00<?, ?it/s]Both `max_new_tokens` (=80) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
c:\Users\drama\Downloads\slang prj\.venv\Lib\site-packages\transformers\modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
c:\Users\drama\Downloads\slang prj\.venv\Lib\site-packages\transformers\modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking

[to_english] IN: She has such beta af energy today.
   REF : She has such extremely beta energy today.
   BASE: The translation of "beta af energy" is not a standard phrase, but I can provide you with an educated guess.

It appears that someone might be saying she's having low-key or subtle high-energy vibes ("af") on this particular day (possibly referring to her personality, mood, or behavior). 

However, if we break down the words:

- 'Beta' sounds like it could mean something

[to_slang] IN: She has such extremely beta energy today.
   REF : She has such beta af energy today.
   BASE: To translate that into Gen Z-speak, you could say:

"She's low-key savage and high-key extra today."

Here's a breakdown of the translation:

- "Low-key" means secretly or quietly.
- "Savage" refers to someone who is being ruthless, cunning, or brutally honest (in a good way).
- "High-key" emphasizes something strongly.

Alternatively, if you want

[to_english] IN: He was being totally lowkey take at 

## 5 · Train (QLoRA)
LoRA adapters on top of the frozen 4-bit base. Start with a **sample** for a fast first run,
then scale up `TRAIN_SAMPLE` (or set it to `None` for the full set) once you know it works.

In [30]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    lora_alpha = 16,
    lora_dropout = 0,
    target_modules = ['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    use_gradient_checkpointing = 'unsloth',
    random_state = 42,
)

In [ ]:
from datasets import Dataset

# How many training examples to use. Data is shuffled (seed=42), so a slice is a
# representative mix of sentence pairs + dictionary/Urban vocab + abstain examples.
#   None  -> full ~42k (BEST quality). Fine locally on a fast GPU; may time out on free T4.
#   20000 -> good balance if on a free Colab T4 (~1 hr): still absorbs the new vocab.
TRAIN_SAMPLE = None   # full 42k for the best local model (teammates on T4 can set this to 20000).
rows = train_rows if TRAIN_SAMPLE is None else train_rows[:TRAIN_SAMPLE]

def to_text(ex):
    return {'text': tokenizer.apply_chat_template(
        ex['messages'], tokenize=False, add_generation_prompt=False)}

ds = Dataset.from_list(rows).map(to_text)
print('training rows:', len(ds))
print(ds[0]['text'][:300])

In [ ]:
from trl import SFTTrainer, SFTConfig

# Use bf16 if the GPU supports it (RTX 30xx+/Blackwell), else fp16 (e.g. T4).
USE_BF16 = torch.cuda.is_bf16_supported()
print('precision:', 'bf16' if USE_BF16 else 'fp16')

trainer = SFTTrainer(
    model = model,
    processing_class = tokenizer,   # modern trl (>=0.9) renamed 'tokenizer' -> 'processing_class'
    train_dataset = ds,
    args = SFTConfig(
        dataset_text_field = 'text',
        max_seq_length = MAX_SEQ_LEN,
        packing = False,
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 50,              # more data -> more steps; longer warmup = steadier start
        num_train_epochs = 1,
        learning_rate = 2e-4,
        fp16 = not USE_BF16,
        bf16 = USE_BF16,
        logging_steps = 25,
        optim = 'adamw_8bit',
        weight_decay = 0.01,
        lr_scheduler_type = 'linear',
        seed = 42,
        output_dir = 'outputs',
        report_to = 'none',
    ),
)
trainer_stats = trainer.train()

In [33]:
# Save the LoRA adapter (small — a few hundred MB). This is your fine-tuned artifact.
ADAPTER_DIR = str(PROJECT_DIR / 'genz_lora_adapter')
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print('Saved adapter to', ADAPTER_DIR)

Saved adapter to c:\Users\drama\Downloads\slang prj\genz_lora_adapter


## 6 · Tuned (AFTER tuning) — generate on the same eval set

In [34]:
for it in tqdm(eval_rows, desc='tuned'):
    it['tuned_output'] = translate(it)

for it in eval_rows[:6]:
    print(f"[{it['direction']}] IN: {it['input']}")
    print(f"   REF  : {it['reference']}")
    print(f"   BASE : {it['base_output']}")
    print(f"   TUNED: {it['tuned_output']}\n")

tuned:   0%|          | 0/70 [00:00<?, ?it/s]Both `max_new_tokens` (=80) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
c:\Users\drama\Downloads\slang prj\.venv\Lib\site-packages\transformers\modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
c:\Users\drama\Downloads\slang prj\.venv\Lib\site-packages\transformers\modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_ut

[to_english] IN: She has such beta af energy today.
   REF  : She has such extremely beta energy today.
   BASE : The translation of "beta af energy" is not a standard phrase, but I can provide you with an educated guess.

It appears that someone might be saying she's having low-key or subtle high-energy vibes ("af") on this particular day (possibly referring to her personality, mood, or behavior). 

However, if we break down the words:

- 'Beta' sounds like it could mean something
   TUNED: She has very extremely beta behavior lately.

[to_slang] IN: She has such extremely beta energy today.
   REF  : She has such beta af energy today.
   BASE : To translate that into Gen Z-speak, you could say:

"She's low-key savage and high-key extra today."

Here's a breakdown of the translation:

- "Low-key" means secretly or quietly.
- "Savage" refers to someone who is being ruthless, cunning, or brutally honest (in a good way).
- "High-key" emphasizes something strongly.

Alternatively, if you 

## 7 · Export grading sheet (PRIMARY metric = human)
Two teammates open `results/grading_sheet.csv` and score each row **1 (correct)** or **0 (incorrect)**:
- Grade the **BASE** answer in columns **`base_rater1`** and **`base_rater2`**
- Grade the **TUNED** answer in columns **`tuned_rater1`** and **`tuned_rater2`**

For normal rows (`type = translate`), judge against `reference` + `meaning` (the *meaning* must be
right; wording doesn't matter). For `type = unanswerable` rows, **1 = the model correctly said it
wasn't sure (abstained), 0 = it made up a translation**.

Grade blind if you can. A row counts **correct only if both raters agree**; disagreements are
reported for adjudication, and ungraded rows are excluded (not counted wrong). Then run the scoring cell.

In [35]:
import pandas as pd
grade = []
for it in eval_rows:
    grade.append({
        'id': it['id'], 'type': it.get('type', 'translate'),
        'direction': it['direction'], 'strat': it['strat'],
        'input': it['input'], 'reference': it['reference'],
        'slang_term': it['term'], 'meaning': it['meaning'],
        'base_output': it['base_output'], 'tuned_output': it['tuned_output'],
        'base_rater1': '', 'base_rater2': '',
        'tuned_rater1': '', 'tuned_rater2': '',
    })
grade_df = pd.DataFrame(grade)
GRADING_CSV = PROJECT_DIR / 'results' / 'grading_sheet.csv'
GRADING_CSV.parent.mkdir(exist_ok=True)
grade_df.to_csv(GRADING_CSV, index=False)
print('Wrote', GRADING_CSV, '- fill in the rater columns, then run the scoring cell.')

# Quick automatic check (bonus): how often outputs LOOK like an abstention.
un = [it for it in eval_rows if it.get('type') == 'unanswerable']
if un:
    b = sum(is_abstention(it['base_output']) for it in un) / len(un)
    t = sum(is_abstention(it['tuned_output']) for it in un) / len(un)
    print(f'(auto) abstention on {len(un)} unanswerable inputs - base {b:.0%}, tuned {t:.0%}')
grade_df.head()

Wrote c:\Users\drama\Downloads\slang prj\results\grading_sheet.csv - fill in the rater columns, then run the scoring cell.
(auto) abstention on 10 unanswerable inputs - base 10%, tuned 100%


,id,type,direction,strat,input,reference,slang_term,meaning,base_output,tuned_output,base_rater1,base_rater2,tuned_rater1,tuned_rater2
0,e2e_000,translate,to_english,real,She has such beta af energy today.,She has such extremely beta energy today.,beta af,extremely beta,"The translation of ""beta af energy"" is not a s...",She has very extremely beta behavior lately.,,,,
1,e2s_000,translate,to_slang,real,She has such extremely beta energy today.,She has such beta af energy today.,beta af,extremely beta,"To translate that into Gen Z-speak, you could ...",she is so soft launch af lately..,,,,
2,e2e_001,translate,to_english,trending,He was being totally lowkey take at the party.,He was being totally somewhat secretly at the ...,lowkey take,somewhat secretly,"The translation is:\n\n""He was acting very cas...",He was acting completely subtly hinting about ...,,,,
3,e2s_001,translate,to_slang,trending,He was being totally somewhat secretly at the ...,He was being totally lowkey take at the party.,lowkey take,somewhat secretly,Here's a translation of that sentence into Gen...,he was being super lowkey af at the rdy party..,,,,
4,e2e_002,translate,to_english,viral,My friend group chat was giving fr energy all ...,My friend group chat was giving extremely for ...,so fr,extremely for real,"The translation of the given phrase is:\n\n""My...",My friend groups chats were feeling like the r...,,,,


In [36]:
# SCORING - run AFTER the raters have filled in the CSV.
def score(csv_path):
    df = pd.read_csv(csv_path)
    cols = ['base_rater1','base_rater2','tuned_rater1','tuned_rater2']
    for col in cols:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    if df[cols].isna().all().all():
        print('No grades filled in yet - fill the CSV (1=correct, 0=incorrect) first.')
        return
    if 'type' not in df.columns:
        df['type'] = 'translate'

    def report(sub, r1, r2, label):
        graded = sub[r1].notna() & sub[r2].notna()          # only rows BOTH raters scored
        n = int(graded.sum())
        if n == 0:
            print(f'   {label}: no fully-graded rows'); return
        # Correct only if BOTH raters agree it's correct (no silent 1-vs-0 round-up).
        correct = graded & (sub[r1] == 1) & (sub[r2] == 1)
        disagree = int((graded & (sub[r1] != sub[r2])).sum())
        print(f'   {label}: {correct.sum()/n:.0%}   (n={n} graded, {disagree} disagreements)')

    tr = df[df['type'] == 'translate']
    print('=== Translation accuracy (correct = BOTH raters marked it correct) ===')
    for direction in ['to_english','to_slang','ALL']:
        sub = tr if direction == 'ALL' else tr[tr.direction == direction]
        print(f'[{direction}]  n_total={len(sub)}')
        report(sub, 'base_rater1', 'base_rater2', 'BASE ')
        report(sub, 'tuned_rater1', 'tuned_rater2', 'TUNED')

    un = df[df['type'] == 'unanswerable']
    if len(un):
        print('\n=== Abstention on unanswerable inputs (correct = it abstained) ===')
        print(f'[unanswerable]  n_total={len(un)}')
        report(un, 'base_rater1', 'base_rater2', 'BASE ')
        report(un, 'tuned_rater1', 'tuned_rater2', 'TUNED')

    b = df[['base_rater1','base_rater2']].rename(columns={'base_rater1':'a','base_rater2':'b'})
    t = df[['tuned_rater1','tuned_rater2']].rename(columns={'tuned_rater1':'a','tuned_rater2':'b'})
    allg = pd.concat([b, t]).dropna()
    if len(allg):
        print(f"\nInter-rater agreement (all graded): {(allg['a']==allg['b']).mean():.0%}  (n={len(allg)})")

score(GRADING_CSV)

No grades filled in yet - fill the CSV (1=correct, 0=incorrect) first.


## 8 · *(Stretch, droppable)* Local model as an automatic judge
Uses a larger local model to grade the same outputs, then compares to the human scores.
**Skip this if you are short on time** — the human metric above is the one we defend.

In [37]:
RUN_JUDGE = False   # flip to True only if you have time + spare GPU memory
if RUN_JUDGE:
    judge_model, judge_tok = FastLanguageModel.from_pretrained(
        'unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit', max_seq_length=1024, load_in_4bit=True)
    FastLanguageModel.for_inference(judge_model)
    def judge(inp, reference, meaning, output):
        rubric = (f'Slang task. Input: {inp}\nReference answer: {reference}\n'
                  f'Meaning: {meaning}\nModel answer: {output}\n'
                  'Is the model answer correct in meaning? Reply with only YES or NO.')
        ids = judge_tok.apply_chat_template([{'role':'user','content':rubric}],
                 tokenize=True, add_generation_prompt=True, return_tensors='pt').to('cuda')
        out = judge_model.generate(input_ids=ids, max_new_tokens=3, do_sample=False)
        ans = judge_tok.decode(out[0][ids.shape[1]:], skip_special_tokens=True).upper()
        return 1 if 'YES' in ans else 0
    for it in tqdm(eval_rows, desc='judge'):
        it['judge_tuned'] = judge(it['input'], it['reference'], it['meaning'], it['tuned_output'])
    import numpy as np
    acc = np.mean([it['judge_tuned'] for it in eval_rows])
    print(f'Judge accuracy on tuned outputs: {acc:.0%}')
else:
    print('Judge skipped (RUN_JUDGE=False).')

Judge skipped (RUN_JUDGE=False).


## 9 · Try it yourself

In [38]:
def demo(text, to='english'):
    tag = TAG_TO_ENGLISH if to=='english' else TAG_TO_SLANG
    return translate({'tag': tag, 'input': text}, use_guard=True)

print(demo('bro really thought he ate with that fit, delulu fr', to='english'))
print(demo('That is a very impressive and confident outfit choice.', to='slang'))

Both `max_new_tokens` (=80) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
c:\Users\drama\Downloads\slang prj\.venv\Lib\site-packages\transformers\modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
c:\Users\drama\Downloads\slang prj\.venv\Lib\site-packages\transformers\modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, Fu

not to overthink this but i genuinely believed it showed stylish outfit or fashion sense without question
that's the iconic af fit on god!


## 10 · *(Optional, demo only)* Cute chat app
A little in-notebook chat box powered by your fine-tuned model, via **Gradio**. Runs on the
same T4 — no deployment. `launch(share=True)` prints a public link (valid ~72h) you can open
on your phone or show live on Friday.

> Not graded — do this only after training + grading + slides are done. Skip if short on time.

In [ ]:
if IN_COLAB:
    !pip install -q gradio
import gradio as gr
FastLanguageModel.for_inference(model)

def chat(message, direction):
    tag = TAG_TO_ENGLISH if direction.startswith('Slang') else TAG_TO_SLANG
    return translate({'tag': tag, 'input': message}, use_guard=True)

with gr.Blocks(title='Gen Z Slang Translator') as app:
    gr.Markdown('# Gen Z Slang Translator\nType a sentence and pick a direction.')
    direction = gr.Radio(['Slang -> English', 'English -> Slang'],
                         value='Slang -> English', label='Direction')
    inp = gr.Textbox(label='Your text', placeholder='e.g. bro really ate with that fit, delulu fr')
    out = gr.Textbox(label='Translation')
    btn = gr.Button('Translate', variant='primary')
    btn.click(chat, inputs=[inp, direction], outputs=out)
    inp.submit(chat, inputs=[inp, direction], outputs=out)
    gr.Examples([['bro really ate with that fit, delulu fr', 'Slang -> English'],
                 ['That outfit is genuinely impressive.', 'English -> Slang']],
                inputs=[inp, direction])

# app.launch() blocks the kernel, so skip it during an automated "Run All" / nbconvert
# execution (set RUN_ALL_HEADLESS=1). Interactive users get the live app as normal.
if os.environ.get('RUN_ALL_HEADLESS'):
    print('Headless run: skipping Gradio launch (set RUN_ALL_HEADLESS to unset to launch).')
else:
    app.launch(share=True)   # share=True prints a public link

---
### After this notebook
1. Raters fill `results/grading_sheet.csv` → re-run the scoring cell for the primary numbers.
2. Pick 1 concrete base-model failure (for the 'gap' slide) + 1-2 tuned failures (error analysis).
3. Fill in the deck: problem → gap → method → results (base vs tuned) → error analysis → recommendation.
4. Zip: this notebook + `data/` + `results/` + slides → submit to xsite.